In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Utility Functions and Classes

This section contains the implementations of utility functions and classes
used in this book. A cell marked `#@save` is copied into the `d2l` package,
so that any section can call its functions as `d2l.plot`, `d2l.set_figsize`,
and so on.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
import collections
import inspect
from IPython import display
from matplotlib import ticker
from matplotlib_inline import backend_inline

## Hyperparameter Utilities

In [ ]:
@d2l.add_to_class(d2l.HyperParameters)
def save_hyperparameters(self, ignore=None):
    """Save function arguments into instance attributes."""
    ignore = [] if ignore is None else ignore
    frame = inspect.currentframe().f_back
    _, _, _, local_vars = inspect.getargvalues(frame)
    self.hparams = {k:v for k, v in local_vars.items()
                    if k not in set(ignore+['self']) and not k.startswith('_')}
    for k, v in self.hparams.items():
        setattr(self, k, v)

## Progress and Plotting

Most plots in this book come from `d2l.plot`, which draws one or more curves
in one set of axes. The first cell sets the format and size of a figure
(`use_svg_display`, `set_figsize`) and the labels, ranges, and scales of its
axes (`set_axes`). On a logarithmic axis, `set_log_ticks` labels all major
ticks in one format.

In [ ]:
def use_svg_display():
    """Use the svg format to display a plot in Jupyter."""
    backend_inline.set_matplotlib_formats('svg')

def set_figsize(figsize=(3.5, 2.5)):
    """Set the figure size for matplotlib."""
    use_svg_display()
    d2l.plt.rcParams['figure.figsize'] = figsize

def set_log_ticks(axis):
    """Label all major ticks of a log axis in one format."""
    lo, hi = axis.get_view_interval()
    if 0 < lo and hi < 100 * lo:  # Under two decades: ticks at 1, 2, 5
        axis.set_major_locator(ticker.LogLocator(subs=(1, 2, 5)))
    ticks = axis.get_major_locator().tick_values(lo, hi)
    plain = all(1e-4 <= t < 1e5 for t in ticks if lo <= t <= hi)

    def label(v, _):
        if plain:  # 0.001, 0.01, ..., 10000
            return f'{v:g}'
        m, k = f'{v:.3e}'.split('e')  # 2.500e-05 -> '2.500', '-05'
        m, power = f'{float(m):g}', f'10^{{{int(k)}}}'  # '2.5', '10^{-5}'
        return f'${power}$' if m == '1' else f'${m}\\times{power}$'

    axis.set_major_formatter(ticker.FuncFormatter(label))
    axis.set_minor_formatter(ticker.NullFormatter())


def set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend,
             legend_loc=None):
    """Set the axes for matplotlib. legend_loc is a matplotlib legend
    location, or 'outside' for beside the axes, clear of the curves."""
    axes.set_xlabel(xlabel), axes.set_ylabel(ylabel)
    axes.set_xscale(xscale), axes.set_yscale(yscale)
    axes.set_xlim(xlim),     axes.set_ylim(ylim)
    for axis, scale in ((axes.xaxis, xscale), (axes.yaxis, yscale)):
        if scale == 'log':
            set_log_ticks(axis)
    if legend and legend_loc == 'outside':
        axes.legend(legend, loc='center left', bbox_to_anchor=(1, 0.5))
    elif legend:
        axes.legend(legend, loc=legend_loc)  # None: matplotlib's best spot
    axes.grid()

Most of the code of `plot` brings its inputs into the form of a list of
curves.

In [ ]:

def plot(X, Y=None, xlabel=None, ylabel=None, legend=None, xlim=None,
         ylim=None, xscale='linear', yscale='linear',
         fmts=('-', 'm--', 'g-.', 'r:'), figsize=(3.5, 2.5), axes=None,
         legend_loc=None):
    """Plot data points."""
    legend = [] if legend is None else legend

    def has_one_axis(X):  # True if X (tensor or list) has 1 axis
        return (hasattr(X, "ndim") and X.ndim == 1 or isinstance(X, list)
                and not hasattr(X[0], "__len__"))
    
    if has_one_axis(X): X = [X]
    if Y is None:
        X, Y = [[]] * len(X), X
    elif has_one_axis(Y):
        Y = [Y]
    if len(X) != len(Y):
        X = X * len(Y)
        
    set_figsize(figsize)
    if axes is None:
        axes = d2l.plt.gca()
    axes.cla()
    # Extend the formats with new colors so that no curve is dropped
    fmts = list(fmts) + [f'C{i % 10}' + ('-', '--', '-.', ':')[i % 4]
                         for i in range(len(fmts), len(Y))]
    for x, y, fmt in zip(X, Y, fmts):
        axes.plot(x, y, fmt) if len(x) else axes.plot(y, fmt)
    set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend,
             legend_loc)

The `ProgressBoard` of that section draws on the calling thread.
`show` renders all curves into a new figure and replaces the previous frame
of the same board in the notebook output. In the headless book build only the
final call renders, so an executed notebook records one figure per board.
Losses go on the left axis and accuracies on a second axis on the right, in
which case the legend moves above the plot so that it covers neither set of
curves. Validation points, and any curve that has a single point, are marked
with dots, so that a one-epoch run is still visible.

In [ ]:

import os
from matplotlib.figure import Figure

@d2l.add_to_class(d2l.ProgressBoard)
def show(self, final=True):
    """Redraw all curves; the book build renders only the final call."""
    if not self.display or (
            not final and os.environ.get('D2L_NB_CAPTURE') == '1'):
        return
    d2l.use_svg_display()
    fig = Figure(figsize=self.figsize)
    self._plot(fig.add_subplot())
    if self.handle is None:
        self.handle = display.display(fig, display_id=True)
    else:
        self.handle.update(fig)

@d2l.add_to_class(d2l.ProgressBoard)
def _plot(self, ax):
    """Losses on the left axis, accuracies on a right one."""
    acc_ax = None
    for i, (label, (xs, ys)) in enumerate(self.data.items()):
        target = ax
        if label.endswith('acc'):
            acc_ax = acc_ax or ax.twinx()
            target = acc_ax
        # Mark validation points, and any curve that has a single point
        marked = label.startswith('val') or len(xs) == 1
        target.plot(xs, ys, ('-', '--', '-.', ':')[i % 4], color=f'C{i % 10}',
                    marker='o' if marked else None, markersize=3,
                    clip_on=not marked, label=label)
    ax.set_xlabel(self.xlabel), ax.set_ylabel(self.ylabel)
    ax.set_xscale(self.xscale), ax.set_yscale(self.yscale)
    if self.xlim: ax.set_xlim(self.xlim)
    if self.ylim: ax.set_ylim(self.ylim)
    if self.yscale == 'log': d2l.set_log_ticks(ax.yaxis)
    if acc_ax is not None: acc_ax.set_ylabel('accuracy')
    lines = list(ax.get_lines()) + list(acc_ax.get_lines() if acc_ax else [])
    labels = [line.get_label() for line in lines]
    if acc_ax is not None:  # Above the axes: no curve of either axis is hidden
        ax.legend(lines, labels, loc='lower center', ncol=len(lines),
                  bbox_to_anchor=(0.5, 1), frameon=False)
    elif lines:
        ax.legend(lines, labels)
    ax.grid()

`show_trace_2d` draws the iterates of a two-dimensional update rule, as
`d2l.train_2d` of that section returns them, over the contour lines of
the objective `f`. The window covers the start $(-5, -2)$ and the minimum
at the origin, and widens to fit the trace. Both axes share one scale
unless the trace has diverged along one of them.

In [ ]:
def show_trace_2d(f, results):
    """Show the trace of 2D variables over the contours of f."""
    x1s, x2s = zip(*[(float(a), float(b)) for a, b in results])
    grids = []  # A window around start and minimum, widened to fit the trace
    for (lo, hi), xs in zip([(-5.5, 1.0), (-3.0, 3.0)], (x1s, x2s)):
        lo, hi = min(lo, min(xs) - 0.5), max(hi, max(xs) + 0.5)
        grids.append(d2l.arange(lo, hi, (hi - lo) / 100))
    x1, x2 = d2l.meshgrid(*grids, indexing='ij')
    width, height = (float(g[-1] - g[0]) for g in grids)
    equal = 0.25 < height / width < 4  # Equal scales unless the trace diverged
    d2l.set_figsize((3, 2.8) if equal else (3.5, 2.5))
    d2l.plt.contour(x1, x2, f(x1, x2), colors='0.6', linewidths=0.8)
    d2l.plt.plot(x1s, x2s, '-o', color='C1', markersize=4)
    d2l.plt.xlabel('$x_1$')
    d2l.plt.ylabel('$x_2$')
    if equal:
        d2l.plt.gca().set_aspect('equal')

## Reinforcement Learning Utilities

In [ ]:

def plot_curves(curves, xlabel, ylabel, smooth=1, reference=None,
                ylim=None, x=None, xscale='linear', yscale='linear',
                band='range'):
    """One panel per call. `curves` maps a label to an array of shape
    [seed, step] (or [step] for a single seed); plots the seed median and a
    shaded band over the seeds: min to max (`band='range'`) or the
    interquartile range (`band='iqr'`, the better summary from five seeds
    on). `smooth`, if greater than 1, replaces each point by the mean of the
    `smooth` steps that end there, before the seed statistics; the first
    `smooth - 1` steps lack a full window and are dropped. `x` gives the
    position of each step (default 0, 1, 2, ...). `reference`, if given,
    draws a dashed horizontal line; `ylim` fixes the y-axis range; `xscale`
    and `yscale` take matplotlib's scale names, e.g. 'log'."""
    set_figsize((6, 4))
    for name, values in curves.items():
        values = np.atleast_2d(values)
        xs = np.arange(values.shape[1]) if x is None else np.asarray(x)
        if smooth > 1:
            kernel = np.ones(smooth) / smooth
            values = np.stack([np.convolve(v, kernel, 'valid')
                               for v in values])
            xs = xs[smooth - 1:]      # each window sits at its last step
        line, = plt.plot(xs, np.median(values, axis=0), label=name)
        if values.shape[0] > 1:
            lo, hi = ((values.min(axis=0), values.max(axis=0))
                      if band == 'range' else
                      np.percentile(values, [25, 75], axis=0))
            plt.fill_between(xs, lo, hi, alpha=0.2, color=line.get_color())
    if reference is not None:
        plt.axhline(reference, linestyle='--', color='gray')
    if ylim is not None:
        plt.ylim(ylim)
    plt.xscale(xscale)
    plt.yscale(yscale)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()

def plot_hist(samples, xlabel, ylabel, bins=60, vlines=()):
    """Overlaid histograms of `samples` (label -> values), dashed at `vlines`."""
    set_figsize((6, 4))
    for name, x in samples.items():
        plt.hist(x, bins=bins, alpha=0.5, label=name)
    for v in vlines:
        plt.axvline(v, linestyle='--', color='black')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()

def show_grid(desc, values, policy=None, titles=None,
              diverging=False, fmt='{:.2f}'):
    """A gridworld in the style of the chapter's figures. Each cell is
    shaded by `values` (white to blue, or with `diverging=True` red to blue
    around zero, for differences) and shows its state index (top left), its
    value (bottom left) and an arrow for the action of `policy`, if given.
    Holes are grey and the goal green; S, H and G sit at the bottom right.
    The grid shape is read from `desc` (e.g. `env.unwrapped.desc`).
    `values` and `policy` may each be a single frame, shape (num_states,),
    or a sequence of frames, shape (num_frames, num_states); `titles`, if
    given, labels each frame."""
    h, w = desc.shape
    values = np.atleast_2d(values).reshape(-1, h * w)
    if policy is not None:
        policy = np.atleast_2d(policy).reshape(-1, h * w)
    if diverging:
        top = max(np.abs(values).max(), 1e-12)
        cmap, norm = plt.get_cmap('RdBu'), plt.Normalize(-top, top)
    else:
        cmap = plt.get_cmap('Blues')
        norm = plt.Normalize(min(0.0, values.min()),
                             max(values.max(), 1e-12))
    move = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}   # (dx, dy)
    fig, axes = plt.subplots(1, len(values), squeeze=False,
                             figsize=(3.6 * len(values), 3.6))
    for k, ax in enumerate(axes[0]):
        for s in range(h * w):
            y, x = divmod(s, w)
            cell, v = desc[y, x].decode(), values[k, s]
            fill = {'H': ('#565D66', 0.55), 'G': ('#2CA02C', 0.35)}.get(
                cell, (cmap(norm(v)), 1.0))
            ax.add_patch(plt.Rectangle((x, y), 1, 1, fc=fill[0],
                                       alpha=fill[1], ec='black', lw=1.0))
            shade = np.abs(norm(v) - 0.5) * 2 if diverging else norm(v)
            ink = 'white' if cell == 'H' or (cell != 'G' and shade > 0.6) \
                else 'black'
            ax.text(x + 0.07, y + 0.07, str(s), ha='left', va='top',
                    fontsize=9, color=ink)
            if cell not in 'HG' or np.abs(v) > 1e-9:   # terminal: 0
                ax.text(x + 0.07, y + 0.93, fmt.format(v), ha='left',
                        va='bottom', fontsize=10, color=ink)
            if cell in 'SHG':
                ax.text(x + 0.82, y + 0.80, cell, ha='center', va='center',
                        fontsize=13, fontweight='bold', color=ink)
            if policy is not None and cell not in 'HG':
                dx, dy = move[int(policy[k, s])]
                ax.annotate('', xy=(x + 0.5 + 0.22 * dx, y + 0.45 + 0.22 * dy),
                            xytext=(x + 0.5 - 0.22 * dx,
                                    y + 0.45 - 0.22 * dy),
                            arrowprops=dict(arrowstyle='-|>', color=ink,
                                            lw=1.6, mutation_scale=14))
        ax.set_xlim(0, w)
        ax.set_ylim(h, 0)
        ax.set_aspect('equal')
        ax.axis('off')
        if titles is not None:
            ax.set_title(titles[k])
    fig.tight_layout()

def run_seeds(train, num_seeds, **kwargs):
    """Run train(seed, **kwargs), a generator of curve points, per seed."""
    return np.array([list(train(seed, **kwargs)) for seed in range(num_seeds)])

def on_grid(runs, grid, col, smooth=1):
    """Resample per-episode rows (env step, value, ...) onto an env-step
    grid. For each run, drop the rows whose column `col` is NaN, average
    the column over the last `smooth` episodes, and interpolate linearly at
    the grid points; returns [run, grid point], the input of plot_curves."""
    out = []
    for r in runs:
        r = np.asarray(r, dtype=float)
        r = r[~np.isnan(r[:, col])]
        v = np.convolve(r[:, col], np.ones(smooth) / smooth, 'valid')
        out.append(np.interp(grid, r[smooth - 1:, 0], v))
    return np.stack(out)

`run_seeds` runs a training generator once per seed and stacks the curves it yields into one array of shape (seed, step, ...), the input `plot_curves` expects. Off-policy agents report once per episode, and seeds finish different numbers of episodes; `on_grid` puts such per-episode records on a common grid of environment steps, so that they too can be stacked and plotted.

The JAX implementations of the reinforcement-learning agents act one state at a time and update on batches whose length changes from one batch to the next. Compiled code is keyed on the structure of the network, the shapes of its inputs and the optimizer, so three helpers keep the number of compilations small. `forward_np` compiles one forward pass per network structure and reads the network's current parameters at each call, so optimizer steps need no recompilation. `_pad` and `_bucket` pad a batch to the next power of two (at least 64 rows), so a compiled update is reused across batches of different lengths; `padded` pads several arrays of one batch together and adds a mask that marks the real rows, so that an update can leave the padding out of its means. `shared_adam` keeps one `optax.adam` per learning rate, so agents created with the same learning rate share their compiled updates, and `plain_sgd` does the same for plain SGD, which some experiments use in place of Adam.

In [ ]:
def plain_sgd(net, lr):
    """A plain SGD optimizer over the parameters of `net`."""
    return torch.optim.SGD(net.parameters(), lr=lr)

## Feature Networks for Image Scores

The generative chapters score images in the features of a small CIFAR-10 classifier (that section). `load_cifar10_arrays` returns the CIFAR-10 training and test images, scaled to $[-1, 1]$, with their labels. `FeatureCNN` is a classifier of three convolution blocks, each with batch normalization, ReLU and $2 \times 2$ max pooling, followed by global average pooling and a linear head; its `features` method returns the pooled features, and its `resolution` argument sets the size to which it first resizes its inputs, low-pass filtering them when it shrinks them. `train_feature_cnn` trains it with Adam under a one-cycle learning-rate schedule on random horizontal flips and, optionally, random shifts.

In [ ]:

def load_cifar10_arrays():
    """CIFAR-10 on the GPU if there is one, pixel values scaled to [-1, 1].
    Returns train_X, train_y, test_X, test_y."""
    import pickle, warnings  # not in the d2l library preamble
    d2l.DATA_HUB['cifar10'] = (
        'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz',
        '874905e36347c8536514d0a26261acf3bff89bc7')
    cifar_dir = d2l.download_extract('cifar10', 'cifar-10-batches-py')
    def load_batch(name):
        with open(f'{cifar_dir}/{name}', 'rb') as f, warnings.catch_warnings():
            warnings.simplefilter('ignore')   # legacy NumPy pickle format
            batch = pickle.load(f, encoding='bytes')
        X = torch.tensor(batch[b'data'], dtype=torch.float32)
        return X.reshape(-1, 3, 32, 32) / 127.5 - 1, torch.tensor(
            batch[b'labels'])
    Xs, ys = zip(*[load_batch(f'data_batch_{i}') for i in range(1, 6)])
    test_X, test_y = load_batch('test_batch')
    device = d2l.try_gpu()
    return (torch.cat(Xs).to(device), torch.cat(ys).to(device),
            test_X.to(device), test_y.to(device))


class FeatureCNN(nn.Module):
    """Three conv blocks, global average pooling, and a linear head. Inputs
    are first resized bilinearly to resolution x resolution."""
    def __init__(self, feature_dim=128, resolution=32):
        super().__init__()
        self.resolution = resolution
        widths = [3, 32, 64, feature_dim]
        self.blocks = nn.ModuleList(
            [nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1),
                           nn.BatchNorm2d(c_out), nn.ReLU(),
                           nn.MaxPool2d(2))
             for c_in, c_out in zip(widths[:-1], widths[1:])])
        self.head = nn.Linear(feature_dim, 10)

    def features(self, x):
        if x.shape[-1] != self.resolution:
            x = F.interpolate(x, size=(self.resolution, self.resolution),
                              mode='bilinear', antialias=True)
        for block in self.blocks:
            x = block(x)
        return x.mean(dim=(2, 3))

    def forward(self, x):
        return self.head(self.features(x))


def train_feature_cnn(cnn, X, y, num_epochs=15, max_lr=0.003,
                      batch_size=256, shift=0, seed=0):
    """Train on random horizontal flips, plus random shifts of up to `shift`
    pixels, with a one-cycle learning rate; end in evaluation mode."""
    gen = torch.Generator(X.device).manual_seed(seed)  # local random streams
    gen_shift = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(cnn.parameters())
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr, total_steps=num_epochs * (len(X) // batch_size),
        cycle_momentum=False)
    h, w = X.shape[2:]
    cnn.train()
    for epoch in range(num_epochs):
        order = torch.randperm(len(X), device=X.device, generator=gen)
        for i in range(0, len(order) - batch_size + 1, batch_size):
            Xb, yb = X[order[i:i + batch_size]], y[order[i:i + batch_size]]
            flip = torch.rand(len(Xb), 1, 1, 1, device=X.device,
                              generator=gen) < 0.5
            Xb = torch.where(flip, Xb.flip(-1), Xb)
            a, b = torch.randint(0, 2 * shift + 1, (2,),
                                 generator=gen_shift).tolist()
            Xb = F.pad(Xb, (shift,) * 4, mode='reflect')[..., a:a + h, b:b + w]
            loss = F.cross_entropy(cnn(Xb), yb)
            opt.zero_grad(), loss.backward(), opt.step(), sched.step()
    cnn.eval()

## Training Utilities

The following legacy helper functions are retained for backward compatibility:

In [ ]:

def load_array(data_arrays, batch_size, is_train=True):
    """Construct a PyTorch data iterator."""
    dataset = torch.utils.data.TensorDataset(*data_arrays)
    return torch.utils.data.DataLoader(dataset, batch_size, shuffle=is_train)

def sgd(params, lr, batch_size):
    """Minibatch stochastic gradient descent."""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

def get_dataloader_workers():
    """Use 4 processes to read the data."""
    return 4

def load_data_fashion_mnist(batch_size, resize=None):
    """Download the Fashion-MNIST dataset and then load it into memory."""
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True)
    return (torch.utils.data.DataLoader(mnist_train, batch_size, shuffle=True,
                                        num_workers=get_dataloader_workers()),
            torch.utils.data.DataLoader(mnist_test, batch_size, shuffle=False,
                                        num_workers=get_dataloader_workers()))

def evaluate_accuracy_gpu(net, data_iter, device=None):
    """Compute the accuracy for a model on a dataset using a GPU."""
    if isinstance(net, nn.Module):
        net.eval()  # Set the model to evaluation mode
        if not device:
            device = next(iter(net.parameters())).device
    # No. of correct predictions, no. of predictions
    metric = d2l.Accumulator(2)

    with torch.no_grad():
        for X, y in data_iter:
            if isinstance(X, list):
                # Required for BERT Fine-tuning (to be covered later)
                X = [x.to(device) for x in X]
            else:
                X = X.to(device)
            y = y.to(device)
            metric.add(d2l.accuracy(net(X), y), d2l.size(y))
    return metric[0] / metric[1]


def train_ch6(net, train_iter, test_iter, num_epochs, lr, device):
    """Train a model with a GPU (defined in Chapter 6)."""
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    net.apply(init_weights)
    print('training on', device)
    net.to(device)
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    loss = nn.CrossEntropyLoss()
    animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs],
                            legend=['train loss', 'train acc', 'test acc'])
    timer, num_batches = d2l.Timer(), len(train_iter)
    for epoch in range(num_epochs):
        # Sum of training loss, sum of training accuracy, no. of examples
        metric = d2l.Accumulator(3)
        net.train()
        for i, (X, y) in enumerate(train_iter):
            timer.start()
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            y_hat = net(X)
            l = loss(y_hat, y)
            l.backward()
            optimizer.step()
            with torch.no_grad():
                metric.add(l * X.shape[0], d2l.accuracy(y_hat, y), X.shape[0])
            timer.stop()
            train_l = metric[0] / metric[2]
            train_acc = metric[1] / metric[2]
            if (i + 1) % (num_batches // 5) == 0 or i == num_batches - 1:
                animator.add(epoch + (i + 1) / num_batches,
                             (train_l, train_acc, None))
        test_acc = evaluate_accuracy_gpu(net, test_iter)
        animator.add(epoch + 1, (None, None, test_acc))
    print(f'loss {train_l:.3f}, train acc {train_acc:.3f}, '
          f'test acc {test_acc:.3f}')
    print(f'{metric[2] * num_epochs / timer.sum():.1f} examples/sec '
          f'on {str(device)}')

In [ ]:
def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5,
                cmap=None, vmin=None, vmax=None):
    """Plot a list of images (cmap colors single-channel images, on the
    color scale from vmin to vmax if given)."""
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        try:
            img = d2l.numpy(img)
        except:
            pass
        ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i])
    return axes

`show_heatmaps` draws matrices as grids of colored cells, one panel per
matrix. Its argument has the shape (rows of panels, columns of panels,
height, width). By default all panels share one color scale and one
colorbar; `cbar='each'` gives every panel its own scale and colorbar, and
`cbar=False` draws none. Ticks mark integer rows and columns, and
`xticklabels` and `yticklabels` name them, e.g., by token. It displays the
confusion matrix of that section and the attention weights
of that section.

In [ ]:
def show_heatmaps(matrices, xlabel, ylabel, titles=None,
                  figsize=(2.5, 2.5), cmap='Reds', cbar=True,
                  xticklabels=None, yticklabels=None):
    """Show heatmaps of matrices; cbar is True (shared), 'each' or False."""
    d2l.use_svg_display()
    matrices = d2l.numpy(matrices)
    num_rows, num_cols, height, width = matrices.shape
    fig, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize,
                                 sharex=True, sharey=True, squeeze=False)
    # All panels share one color scale (one norm) unless each gets its own
    lims = {} if cbar == 'each' else dict(norm=d2l.plt.Normalize(
        matrices.min(), matrices.max()))
    for i, (row_axes, row_matrices) in enumerate(zip(axes, matrices)):
        for j, (ax, matrix) in enumerate(zip(row_axes, row_matrices)):
            pcm = ax.imshow(matrix, cmap=cmap, **lims)
            if cbar == 'each':
                fig.colorbar(pcm, ax=ax, shrink=0.6)
            if i == num_rows - 1:
                ax.set_xlabel(xlabel)
            if j == 0:
                ax.set_ylabel(ylabel)
            if titles:
                ax.set_title(titles[j])
    # Shared axes: setting the ticks of one panel sets them for all
    for axis, n, labels in ((ax.xaxis, width, xticklabels),
                            (ax.yaxis, height, yticklabels)):
        if labels is None:
            axis.set_major_locator(d2l.plt.MaxNLocator(
                'auto', integer=True, steps=[1, 2, 2.5, 5, 10]))
        else:  # Named ticks; names along the x-axis read vertically
            axis.set_ticks(range(n), labels,
                           rotation=90 if axis is ax.xaxis else 0)
    if cbar is True:
        fig.colorbar(pcm, ax=axes, shrink=0.6);

In [ ]:

def linreg(X, w, b):
    """The linear regression model."""
    return d2l.matmul(X, w) + b

def squared_loss(y_hat, y):
    """Squared loss."""
    return (y_hat - d2l.reshape(y, y_hat.shape)) ** 2 / 2

def get_fashion_mnist_labels(labels):
    """Return text labels for the Fashion-MNIST dataset."""
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
                   'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [text_labels[int(i)] for i in labels]

#@tab pytorch, mxnet, tensorflow
class Animator:
    """For plotting data in animation."""
    def __init__(self, xlabel=None, ylabel=None, legend=None, xlim=None,
                 ylim=None, xscale='linear', yscale='linear',
                 fmts=('-', 'm--', 'g-.', 'r:'), nrows=1, ncols=1,
                 figsize=(3.5, 2.5)):
        # Incrementally plot multiple lines
        if legend is None:
            legend = []
        d2l.use_svg_display()
        self.fig, self.axes = d2l.plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [self.axes, ]
        # Use a lambda function to capture arguments
        self.config_axes = lambda: d2l.set_axes(
            self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        # Add multiple data points into the figure
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)

#@tab pytorch, mxnet, tensorflow
class Accumulator:
    """For accumulating sums over `n` variables."""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


#@tab pytorch, mxnet, tensorflow
def accuracy(y_hat, y):
    """Compute the number of correct predictions."""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = d2l.argmax(y_hat, axis=1)
    elif (len(y_hat.shape) > 1 and y_hat.shape[-1] == 1) or y_hat.dtype != y.dtype:
        # Binary classification with float scores (logits or probabilities):
        # threshold at 0 (logits) to get class labels, then reshape to match y.
        y_hat = d2l.astype(y_hat > 0, y.dtype).reshape(y.shape)
    cmp = d2l.astype(y_hat, y.dtype) == y
    return float(d2l.reduce_sum(d2l.astype(cmp, y.dtype)))

In [ ]:

import os
import requests
import zipfile
import tarfile
import hashlib

def download(url, folder='../data', sha1_hash=None):
    """Download a file to folder and return the local filepath."""
    if not url.startswith('http'):
        # For back compatibility
        url, sha1_hash = DATA_HUB[url]
    os.makedirs(folder, exist_ok=True)
    fname = os.path.join(folder, url.split('/')[-1])
    # Check if hit cache
    if os.path.exists(fname) and sha1_hash:
        sha1 = hashlib.sha1()
        with open(fname, 'rb') as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname
    # Download
    import tempfile
    print(f'Downloading {fname} from {url}...')
    r = requests.get(url, stream=True, verify=True)
    fd, tmp = tempfile.mkstemp(dir=folder)
    try:
        with os.fdopen(fd, 'wb') as f:
            f.write(r.content)
        os.replace(tmp, fname)
    except BaseException:
        os.unlink(tmp)
        raise
    return fname

def extract(filename, folder=None):
    """Extract a zip/tar file into folder."""
    import tempfile, shutil
    base_dir = os.path.dirname(filename)
    _, ext = os.path.splitext(filename)
    assert ext in ('.zip', '.tar', '.gz'), 'Only support zip/tar files.'
    opener = zipfile.ZipFile if ext == '.zip' else tarfile.open
    if folder is None:
        folder = base_dir
    tmp = tempfile.mkdtemp(dir=folder)
    try:
        with opener(filename, 'r') as fp:
            fp.extractall(tmp)
        for name in os.listdir(tmp):
            src = os.path.join(tmp, name)
            dst = os.path.join(folder, name)
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            os.rename(src, dst)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

In [ ]:

def download_extract(name, folder=None):
    """Download and extract a zip/tar file."""
    fname = download(name)
    base_dir = os.path.dirname(fname)
    data_dir, ext = os.path.splitext(fname)
    target = os.path.join(base_dir, folder) if folder else data_dir
    # Skip re-extraction if a completion marker exists (extracting many small
    # files is slow and unnecessary when the archive is already unpacked).
    marker = fname + '.extracted'
    if os.path.exists(marker):
        return target
    if ext == '.zip':
        fp = zipfile.ZipFile(fname, 'r')
    elif ext in ('.tar', '.gz'):
        fp = tarfile.open(fname, 'r')
    else:
        assert False, 'Only zip/tar files can be extracted.'
    fp.extractall(base_dir)
    open(marker, 'w').close()
    return target


def tokenize(lines, token='word'):
    """Split text lines into word or character tokens."""
    assert token in ('word', 'char'), 'Unknown token type: ' + token
    return [line.split() if token == 'word' else list(line) for line in lines]

In [ ]:

def evaluate_loss(net, data_iter, loss):
    """Evaluate the loss of a model on the given dataset."""
    metric = d2l.Accumulator(2)  # Sum of losses, no. of examples
    for X, y in data_iter:
        out = net(X)
        y = d2l.reshape(y, out.shape)
        l = loss(out, y)
        metric.add(d2l.reduce_sum(l), d2l.size(l))
    return metric[0] / metric[1]

In [ ]:
def grad_clipping(net, theta):
    """Clip the gradient."""
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if p.requires_grad]
    else:
        params = net.params
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params:
            param.grad[:] *= theta / norm

## Sequence and Attention Utilities

In [ ]:

d2l.DATA_HUB['fra-eng'] = (d2l.DATA_URL + 'fra-eng.zip',
                           '94646ad1522d915e7b0f9296181140edcf86a4f5')


def read_data_nmt():
    """Load the English-French dataset."""
    data_dir = d2l.download_extract('fra-eng')
    with open(os.path.join(data_dir, 'fra.txt'), 'r', encoding='utf-8') as f:
        return f.read()


def preprocess_nmt(text):
    """Preprocess the English-French dataset."""
    def no_space(char, prev_char):
        return char in set(',.!?') and prev_char != ' '

    # Replace non-breaking space with space, and convert uppercase letters to
    # lowercase ones
    text = text.replace('\u202f', ' ').replace('\xa0', ' ').lower()
    # Insert space between words and punctuation marks
    out = [' ' + char if i > 0 and no_space(char, text[i - 1]) else char
           for i, char in enumerate(text)]
    return ''.join(out)


def tokenize_nmt(text, num_examples=None):
    """Tokenize the English-French dataset."""
    source, target = [], []
    for i, line in enumerate(text.split('\n')):
        if num_examples and i > num_examples:
            break
        parts = line.split('\t')
        if len(parts) == 2:
            source.append(parts[0].split(' '))
            target.append(parts[1].split(' '))
    return source, target

    

def truncate_pad(line, num_steps, padding_token):
    """Truncate or pad sequences."""
    if len(line) > num_steps:
        return line[:num_steps]  # Truncate
    return line + [padding_token] * (num_steps - len(line))  # Pad


def build_array_nmt(lines, vocab, num_steps):
    """Transform text sequences of machine translation into minibatches."""
    lines = [vocab[l] for l in lines]
    lines = [l + [vocab['<eos>']] for l in lines]
    array = d2l.tensor([truncate_pad(
        l, num_steps, vocab['<pad>']) for l in lines])
    valid_len = d2l.reduce_sum(
        d2l.astype(array != vocab['<pad>'], d2l.int32), 1)
    return array, valid_len


def load_data_nmt(batch_size, num_steps, num_examples=600):
    """Return the iterator and the vocabularies of the translation dataset."""
    text = preprocess_nmt(read_data_nmt())
    source, target = tokenize_nmt(text, num_examples)
    src_vocab = d2l.Vocab(source, min_freq=2,
                          reserved_tokens=['<pad>', '<bos>', '<eos>'])
    tgt_vocab = d2l.Vocab(target, min_freq=2,
                          reserved_tokens=['<pad>', '<bos>', '<eos>'])
    src_array, src_valid_len = build_array_nmt(source, src_vocab, num_steps)
    tgt_array, tgt_valid_len = build_array_nmt(target, tgt_vocab, num_steps)
    data_arrays = (src_array, src_valid_len, tgt_array, tgt_valid_len)
    data_iter = d2l.load_array(data_arrays, batch_size)
    return data_iter, src_vocab, tgt_vocab

In [ ]:

def sequence_mask(X, valid_len, value=0):
    """Mask irrelevant entries in sequences."""
    maxlen = X.size(1)
    mask = torch.arange((maxlen), dtype=torch.float32,
                        device=X.device)[None, :] < valid_len[:, None]
    X[~mask] = value
    return X

    

class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    """The softmax cross-entropy loss with masks."""
    # `pred` shape: (`batch_size`, `num_steps`, `vocab_size`)
    # `label` shape: (`batch_size`, `num_steps`)
    # `valid_len` shape: (`batch_size`,)
    def __init__(self):
        super().__init__(reduction='none')

    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label)
        weights = sequence_mask(weights, valid_len)
        unweighted_loss = super().forward(pred.permute(0, 2, 1), label)
        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        return weighted_loss
    

def train_seq2seq(net, data_iter, lr, num_epochs, tgt_vocab, device):
    """Train a model for sequence to sequence."""
    def xavier_init_weights(m):
        if type(m) == nn.Linear:
            nn.init.xavier_uniform_(m.weight)
        if type(m) == nn.GRU:
            for param in m._flat_weights_names:
                if "weight" in param:
                    nn.init.xavier_uniform_(m._parameters[param])
    net.apply(xavier_init_weights)
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = MaskedSoftmaxCELoss()
    net.train()
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[10, num_epochs])
    for epoch in range(num_epochs):
        timer = d2l.Timer()
        metric = d2l.Accumulator(2)  # Sum of training loss, no. of tokens
        for batch in data_iter:
            optimizer.zero_grad()
            X, X_valid_len, Y, Y_valid_len = [x.to(device) for x in batch]
            bos = torch.tensor([tgt_vocab['<bos>']] * Y.shape[0],
                               device=device).reshape(-1, 1)
            dec_input = d2l.concat([bos, Y[:, :-1]], 1)  # Teacher forcing
            Y_hat, _ = net(X, dec_input, X_valid_len)
            l = loss(Y_hat, Y, Y_valid_len)
            l.sum().backward()  # Make the loss scalar for `backward`
            d2l.grad_clipping(net, 1)
            num_tokens = Y_valid_len.sum()
            optimizer.step()
            with torch.no_grad():
                metric.add(l.sum(), num_tokens)
        if (epoch + 1) % 10 == 0:
            animator.add(epoch + 1, (metric[0] / metric[1],))
    print(f'loss {metric[0] / metric[1]:.3f}, {metric[1] / timer.stop():.1f} '
          f'tokens/sec on {str(device)}')
    


def predict_seq2seq(net, src_sentence, src_vocab, tgt_vocab, num_steps,
                    device, save_attention_weights=False):
    """Predict for sequence to sequence."""
    # Set `net` to eval mode for inference
    net.eval()
    src_tokens = src_vocab[src_sentence.lower().split(' ')] + [
        src_vocab['<eos>']]
    enc_valid_len = torch.tensor([len(src_tokens)], device=device)
    src_tokens = d2l.truncate_pad(src_tokens, num_steps, src_vocab['<pad>'])
    # Add the batch axis
    enc_X = torch.unsqueeze(
        torch.tensor(src_tokens, dtype=torch.long, device=device), dim=0)
    enc_outputs = net.encoder(enc_X, enc_valid_len)
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    # Add the batch axis
    dec_X = torch.unsqueeze(torch.tensor(
        [tgt_vocab['<bos>']], dtype=torch.long, device=device), dim=0)
    output_seq, attention_weight_seq = [], []
    for _ in range(num_steps):
        Y, dec_state = net.decoder(dec_X, dec_state)
        # We use the token with the highest prediction likelihood as input
        # of the decoder at the next time step
        dec_X = Y.argmax(dim=2)
        pred = dec_X.squeeze(dim=0).type(torch.int32).item()
        # Save attention weights (to be covered later)
        if save_attention_weights:
            attention_weight_seq.append(net.decoder.attention_weights)
        # Once the end-of-sequence token is predicted, the generation of the
        # output sequence is complete
        if pred == tgt_vocab['<eos>']:
            break
        output_seq.append(pred)
    return ' '.join(tgt_vocab.to_tokens(output_seq)), attention_weight_seq

### Embedding Gradients for Small Vocabularies

The gradient of an embedding lookup adds the gradient at each position to
the row of that position's token, and XLA performs these additions with
atomic operations. In a minibatch of text, thousands of positions hold the
same few frequent tokens, such as the bytes of a space and of `e`. Their
additions to the same row wait for one another, and on a GPU the sum can
take tens of times as long as for random tokens. For vocabularies of up to
1,024 tokens, `Embed` computes the same sum as one matrix product of the
one-hot rows of the tokens with the gradients. It is otherwise an
`nnx.Embed`, and the JAX models of the book use it for their token tables.

## A Note on Framework Coverage

The legacy helpers in this section (`evaluate_accuracy`, `train_ch6`,
`train_seq2seq`, `predict_seq2seq`, `MaskedSoftmaxCELoss`) are kept for
parity with the original D2L implementation, which predates the unified
`d2l.Trainer` class introduced in this edition. They are not provided for
JAX, and the PyTorch module includes only the subset used outside the
`Trainer` workflow. If you are reading the JAX tab,
the corresponding chapters use `d2l.Trainer.fit(model, data)` end-to-end;
the corresponding per-batch logic is implemented in
`Trainer.fit_epoch` and the `@d2l.add_to_class` extensions defined
alongside it. The MXNet and TensorFlow tabs additionally retain
`evaluate_accuracy` because some of their earlier-chapter snippets call
it directly; for PyTorch and JAX,
use `Trainer.validation_step` (or its return value) instead.

[Discussions](https://d2l.discourse.group/t/36849)